# Complete Backtesting Workflow

This comprehensive quickstart tutorial demonstrates the complete workflow for backtesting trading strategies using real market data from Databento. We'll then use the data to run a backtest on Nautilus, and visualize the results with our visualizer. The tutorial covers:

1. **Data Fetching**: Download real market data from Databento
2. **Data Processing**: Convert raw data to Nautilus-compatible format
3. **Strategy Development**: Create and configure trading strategies
4. **Backtesting**: Run backtests with realistic market conditions
5. **Visualization**: Analyze results with comprehensive charts

## Prerequisites

Before running this notebook:

1. **Set up your environment** (first time only):
   ```bash
   # Create and activate virtual environment
   python -m venv .venv
   source .venv/bin/activate  # Git Bash/WSL/Linux/macOS
   
   # Install all dependencies
   pip install -r requirements.txt
   pip install -e .
   
   # If you want to run this notebook, install JupyterLab
   pip install jupyterlab
   ```
- Python 3.11 or 3.12 only
- Dependencies installed
- JupyterLab installed

## Setup

First, let's set up our environment and import the necessary modules.

## Step 1: Configure Data Fetching

We'll fetch a small sample of ES (E-mini S&P 500) futures data from Databento. This includes:

- **OHLCV bars**: 1-minute bars for strategy signals
- **Quote ticks**: Top-of-book bid/ask data for realistic execution
- **Trade ticks**: Actual trade data for analysis

**Note**: You'll need a Databento API key. Set it as an environment variable or in your shell:
```bash
export DATABENTO_API_KEY="your-api-key"
```


In [1]:
%%bash
export DATABENTO_API_KEY="your-api-key"

In [1]:
# %% Configuration (custom data pipeline)
# Define all parameters used by our custom fetch / ETL / inspect / CSV steps

# Change to repo root directory to ensure paths work correctly
import os
from pathlib import Path

notebook_dir = Path.cwd()
print(f"Notebook Dir: {notebook_dir}")
repo_root = notebook_dir.parent.parent  # Up two levels from examples/example-01-.../
os.chdir(repo_root)
print(f"✅ Working directory: {Path.cwd()}")

# Dataset and symbols
DATASET = "GLBX.MDP3"            # CME Group market data dataset
SYMBOLS = ["ES.n.0"]             # E-mini S&P 500 continuous contract
STYPE_IN = "continuous"          # Use continuous contract symbol type

# Time range (ISO8601 UTC)
# Note: Using a full week to get enough data for meaningful Sharpe ratio calculation
# Continuous contracts may have overlapping intervals at market close/open boundaries
START_TIME = "2023-12-01T08:30:00Z"   # Market open (first day)
END_TIME   = "2023-12-08T15:01:00Z"   # Market close (last day - full week)

# Schemas to fetch from Databento
SCHEMAS = ["trades", "tbbo", "ohlcv-1m"]

# Instrument and output locations
INSTRUMENT_ID = "ES.n.0.GLBX"     # ES futures on CME Globex
CATALOG_PATH  = "sample_data/catalog"         # Output path for Parquet Data Catalog
INTERIM_PATH  = "interim"         # Output path for extracted CSVs

print("✅ Configuration loaded")
print(f"📊 Dataset: {DATASET}")
print(f"📈 Symbol(s): {SYMBOLS}")
print(f"⏰ Time range: {START_TIME} → {END_TIME}")
print(f"📦 Schemas: {SCHEMAS}")
print(f"🎯 Instrument ID: {INSTRUMENT_ID}")
print(f"📁 Catalog path: {CATALOG_PATH}")

Notebook Dir: c:\SquareKettle\SweTakeHome\examples\example-01-importing-data-and-using-the-backtester
✅ Working directory: c:\SquareKettle\SweTakeHome
✅ Configuration loaded
📊 Dataset: GLBX.MDP3
📈 Symbol(s): ['ES.n.0']
⏰ Time range: 2023-12-01T08:30:00Z → 2023-12-08T15:01:00Z
📦 Schemas: ['trades', 'tbbo', 'ohlcv-1m']
🎯 Instrument ID: ES.n.0.GLBX
📁 Catalog path: sample_data/catalog


## Step 2: Fetch Market Data from Databento

This step downloads the raw market data files from Databento. The data is cached locally to avoid re-downloading on subsequent runs.



In [2]:
# %% Data Pipeline (custom): Fetch raw DBN with our helper
# This cell uses our *custom* Databento fetcher. It does **not** change any downstream logic.
# Inputs assumed from the existing Quickstart config:
#   - DATASET, SYMBOLS, STYPE_IN
#   - SCHEMAS (list like ["mbp-1", "tbbo", "trades"], or whichever you set above)
#   - START_TIME, END_TIME (ISO8601 strings, UTC)
#
# Output:
#   - Raw .dbn.zst files under sample_data/raw/ (content-addressed cache)
from pathlib import Path

# Now we can import from the src directory
from data.fetch_databento import run_fetch

print("🔄 Fetching market data with custom run_fetch() ...")
for schema in SCHEMAS:
    run_fetch(
        dataset=DATASET,
        symbols=SYMBOLS,
        stype_in=STYPE_IN,
        schema=schema,
        start=START_TIME,
        end=END_TIME,
        api_key=None,   # uses DATABENTO_API_KEY env var if set
        force=False,
        notes="quickstart",
        raw_dir="sample_data/raw"  # Specify custom raw directory
    )
print("✅ Raw fetch complete → files cached under ./sample_data/raw/")

🔄 Fetching market data with custom run_fetch() ...


Cache hit: 
sample_data\raw\GLBX.MDP3__ES.n.0__continuous__trades__20231201T083000Z__20231208T150100Z__6879bdecac275bbd495ea580
0ca3bb53a1b8d49a8dc5215854be53208aa444ee.dbn.zst

Cache hit: 
sample_data\raw\GLBX.MDP3__ES.n.0__continuous__tbbo__20231201T083000Z__20231208T150100Z__5c52ef00e5af12cafc193ab85a
679273976d5707b44deef58fff2bf208c84150.dbn.zst

Cache hit: 
sample_data\raw\GLBX.MDP3__ES.n.0__continuous__ohlcv-1m__20231201T083000Z__20231208T150100Z__55c0e63d03d4c35f53b20d
0fd8702c01750070b7e49818fcc541cf944764c6e4.dbn.zst

✅ Raw fetch complete → files cached under ./sample_data/raw/


## Step 3: Process Data into Nautilus Catalog

Now we'll convert the raw Databento files into a Nautilus-compatible Parquet catalog. This step:

1. Creates the ES futures instrument definition
2. Loads and processes the raw data files
3. Writes everything to a structured Parquet catalog
4. Generates a manifest with data summaries


In [3]:
# %% Data Pipeline (custom): Convert raw DBN → Nautilus Parquet catalog
# We construct a Parquet Data Catalog from the fetched DBN artifacts using our ETL helper.
# Output:
#   - ./sample_data/catalog/ with Parquet tables + manifest.json
from pathlib import Path
from data.etl_to_catalog import run_etl

# Collect freshly downloaded DBN artifacts
dbn_paths = sorted([p for p in Path("sample_data/raw").glob("*.dbn.zst")])
if not dbn_paths:
    raise FileNotFoundError("No DBN files found under ./sample_data/raw. Did the fetch step run?")
print(f"🧱 Building Parquet catalog from {len(dbn_paths)} DBN artifact(s)...")

run_etl(
        instrument_id=INSTRUMENT_ID,
        paths=[str(p) for p in dbn_paths],
        catalog_path=CATALOG_PATH,
        overwrite=True,
        raw_manifest_path="sample_data/raw/manifest.json"  # Specify custom manifest path
    )
print("✅ Catalog ready → ./sample_data/catalog/")

🧱 Building Parquet catalog from 3 DBN artifact(s)...


Wrote instrument: ES.n.0.GLBX

Loading 
GLBX.MDP3__ES.n.0__continuous__ohlcv-1m__20231201T083000Z__20231208T150100Z__55c0e63d03d4c35f53b20d0fd8702c01750070
b7e49818fcc541cf944764c6e4.dbn.zst

Loading 
GLBX.MDP3__ES.n.0__continuous__tbbo__20231201T083000Z__20231208T150100Z__5c52ef00e5af12cafc193ab85a679273976d5707b4
4deef58fff2bf208c84150.dbn.zst

Loading 
GLBX.MDP3__ES.n.0__continuous__trades__20231201T083000Z__20231208T150100Z__6879bdecac275bbd495ea5800ca3bb53a1b8d49a
8dc5215854be53208aa444ee.dbn.zst

Catalog ready at sample_data\catalog

{
    'instrument_id': 'ES.n.0.GLBX',
    'classes': {'Bar': 7288, 'QuoteTick': 2196589, 'TradeTick': 2196589},
    'time_bounds_ns': {
        'Bar': {'min': 1701419460000000000, 'max': 1702047660000000000},
        'QuoteTick': {'min': 1701419400163463630, 'max': 1702047659932237367},
        'TradeTick': {'min': 1701419400163463630, 'max': 1702047659932237367}
    },
    'catalog_path': 'sample_data\\catalog'
}

✅ Catalog ready → ./sample_data/catalog/


In [4]:
# %% Data Pipeline (custom): Inspect catalog summary (sanity check)
# Quick one-liner report of what's in our Parquet catalog for this instrument.
from data.inspect_catalog import inspect_catalog_summary

inspect_catalog_summary(catalog_path="sample_data/catalog", instrument_id=INSTRUMENT_ID)

{'manifest_path': 'sample_data\\catalog\\manifest.json',
 'classes': {'Bar': 7288, 'QuoteTick': 2196589, 'TradeTick': 2196589},
 'time_bounds_ns': {'Bar': {'min': 1701419460000000000,
   'max': 1702047660000000000},
  'QuoteTick': {'min': 1701419400163463630, 'max': 1702047659932237367},
  'TradeTick': {'min': 1701419400163463630, 'max': 1702047659932237367}}}

## Step 3.5
Visualize the data in CSV

In [5]:
# %% Data Pipeline (custom): Extract normalized CSVs for quick prototyping/viz
# Creates ./interim/{trades.csv, quotes.csv, bars_for_viz.csv} from the Parquet catalog.
from pathlib import Path
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from data import csv_extract as csvx

out_dir = Path("sample_data/interim"); out_dir.mkdir(parents=True, exist_ok=True)
catalog = ParquetDataCatalog("sample_data/catalog")
# Pull objects from catalog (trades, quotes, OR 1-min bars suitable for the viz)
objs_trades = csvx._catalog_fetch_any(catalog, cls=None, kind="trades", instrument_id=INSTRUMENT_ID)
objs_quotes = csvx._catalog_fetch_any(catalog, cls=None, kind="quotes", instrument_id=INSTRUMENT_ID)
objs_bars   = csvx._catalog_fetch_any(catalog, cls=None, kind="bars",   instrument_id=INSTRUMENT_ID)
# Normalize rows and DataFrames
rows = {}
rows.update(csvx.rows_from_objects(objs_trades, INSTRUMENT_ID))
rows_quotes = csvx.rows_from_objects(objs_quotes, INSTRUMENT_ID)
for k, v in rows_quotes.items(): rows[k].extend(v)
rows_bars = csvx.rows_from_objects(objs_bars, INSTRUMENT_ID)
for k, v in rows_bars.items(): rows[k].extend(v)
dfs = csvx.dataframes_from_rows(rows)
# Write to CSVs expected by the Quickstart viz (filenames kept consistent)
dfs.get("trades",  []).to_csv(out_dir / "trades.csv", index=False)
dfs.get("quotes",  []).to_csv(out_dir / "quotes.csv", index=False)
# Many Quickstarts expect bars_for_viz.csv → map our bars frame to that name
dfs.get("bars",    []).to_csv(out_dir / "bars_for_viz.csv", index=False)
print("✅ CSV extract complete → ./interim/{trades,quotes,bars_for_viz}.csv")

C:\SquareKettle\SweTakeHome\src\data\csv_extract.py:145: UserWarning: [quotes] Detected 6 crossed quotes (bid_price > ask_price). These rows will be kept for traceability — consider filtering before analysis.
  df_q = validate_quotes_df(df_q)


✅ CSV extract complete → ./interim/{trades,quotes,bars_for_viz}.csv


## Step 4: Create Trading Strategy

We'll implement a simple MACD (Moving Average Convergence Divergence) strategy that:

1. Uses 1-minute bars for signal generation
2. Trades on MACD zero-line crossovers
3. Manages position sizing and risk
4. Includes proper position tracking

This strategy serves as a foundation that you can extend with more sophisticated logic.


In [6]:
# Import strategy components
from nautilus_trader.core.message import Event
from nautilus_trader.indicators import MovingAverageConvergenceDivergence
from nautilus_trader.model.position import Position
from nautilus_trader.model.enums import OrderSide, PositionSide, PriceType
from nautilus_trader.model.events import PositionClosed, PositionOpened
from nautilus_trader.trading.strategy import Strategy, StrategyConfig
from nautilus_trader.model.objects import Quantity
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import InstrumentId

class MACDConfig(StrategyConfig):
    """Configuration for MACD strategy."""
    instrument_id: InstrumentId
    bar_type: str
    fast_period: int = 12
    slow_period: int = 26
    trade_size: int = 1  # Number of contracts


class MACDStrategy(Strategy):
    """
    MACD-based strategy that trades on zero-line crossovers.
    
    This strategy:
    - Uses MACD indicator on 1-minute bars
    - Goes long when MACD crosses above zero
    - Goes short when MACD crosses below zero
    - Manages one position at a time
    """

    def __init__(self, config: MACDConfig):
        super().__init__(config=config)
        
        # Initialize MACD indicator
        self.macd = MovingAverageConvergenceDivergence(
            fast_period=config.fast_period, 
            slow_period=config.slow_period, 
            price_type=PriceType.LAST
        )
        
        # Position management
        self.trade_size = Quantity.from_int(config.trade_size)
        self.position: Position | None = None
        self.last_macd_above_zero = None
        
        # Bar type for subscription
        self.bar_type = BarType.from_str(config.bar_type)

        self.custom = []

    def on_start(self):
        """Subscribe to market data on strategy start."""
        self.subscribe_bars(bar_type=self.bar_type)
        self._log.info(f"MACD Strategy started for {self.config.instrument_id}")

    def on_stop(self):
        """Clean up on strategy stop."""
        try:
            self.close_all_positions(self.config.instrument_id)
        finally:
            self.unsubscribe_bars(bar_type=self.bar_type)
        self._log.info("MACD Strategy stopped")

    def on_bar(self, bar):
        """Process incoming bars and generate trading signals."""
        # Update MACD indicator
        self.macd.handle_bar(bar)

        # Wait for indicator to warm up
        if not self.macd.initialized:
            return

        # Check for trading signals
        self.check_signals()

    def on_event(self, event: Event):
        """Handle position events."""
        if isinstance(event, PositionOpened):
            self.position = self.cache.position(event.position_id)
            if self.position is not None:
                self._log.info(f"Position opened: {self.position.side} @ {self.position.avg_px_open}")
        elif isinstance(event, PositionClosed):
            if self.position and self.position.id == event.position_id:
                self._log.info(f"Position closed with PnL: {self.position.realized_pnl}")
                self.position = None

    def check_signals(self):
        """Check for MACD crossover signals."""
        current_macd = self.macd.value
        current_above_zero = current_macd > 0

        # Skip if this is the first reading
        if self.last_macd_above_zero is None:
            self.last_macd_above_zero = current_above_zero
            return

        # Check for crossover
        if self.last_macd_above_zero != current_above_zero:
            if current_above_zero:
                # MACD crossed above zero - go long
                if not self.is_long:
                    if self.is_short:
                        self.close_position(self.position)
                    self.go_long()
            else:
                # MACD crossed below zero - go short
                if not self.is_short:
                    if self.is_long:
                        self.close_position(self.position)
                    self.go_short()

        self.last_macd_above_zero = current_above_zero

    def go_long(self):
        """Enter long position."""
        if self.is_flat:
            order = self.order_factory.market(
                instrument_id=self.config.instrument_id,
                order_side=OrderSide.BUY,
                quantity=self.trade_size
            )
            self.submit_order(order)
            self._log.info(f"Going LONG - MACD: {self.macd.value:.6f}")

    def go_short(self):
        """Enter short position."""
        if self.is_flat:
            order = self.order_factory.market(
                instrument_id=self.config.instrument_id,
                order_side=OrderSide.SELL,
                quantity=self.trade_size
            )
            self.submit_order(order)
            self._log.info(f"Going SHORT - MACD: {self.macd.value:.6f}")

    @property
    def is_flat(self) -> bool:
        """Check if we have no position."""
        return self.position is None

    @property
    def is_long(self) -> bool:
        """Check if we have a long position."""
        return bool(self.position and self.position.side == PositionSide.LONG)

    @property
    def is_short(self) -> bool:
        """Check if we have a short position."""
        return bool(self.position and self.position.side == PositionSide.SHORT)

print("✅ MACD Strategy defined successfully!")
print("📊 Strategy features:")
print("  - MACD zero-line crossover signals")
print("  - Position management")
print("  - Risk controls")
print("  - Comprehensive logging")


✅ MACD Strategy defined successfully!
📊 Strategy features:
  - MACD zero-line crossover signals
  - Position management
  - Risk controls
  - Comprehensive logging


## Step 5: Configure and Run Backtest

Now we'll set up the backtest configuration and execute our strategy. This includes venue setup, data feeds, and the complete backtest run.


In [7]:
from nautilus_trader.backtest.node import BacktestDataConfig
from nautilus_trader.backtest.node import BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.backtest.node import BacktestRunConfig
from nautilus_trader.backtest.node import BacktestVenueConfig
from nautilus_trader.config import ImportableStrategyConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.objects import Quantity
from nautilus_trader.model.data import QuoteTick
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from nautilus_trader.model.data import Bar, QuoteTick, TradeTick
from nautilus_trader.model.identifiers import InstrumentId

# Load catalog and get instrument
catalog = ParquetDataCatalog(CATALOG_PATH)
instruments = catalog.instruments()
instrument = instruments[0]
instrument_id = instrument.id

print(f"🎯 Configuring backtest for instrument: {instrument_id}")

# 1. Venue Configuration
venue = BacktestVenueConfig(
    name=str(instrument_id.venue),  # GLBX
    oms_type="NETTING",
    account_type="MARGIN",
    base_currency="USD",
    starting_balances=["100_000 USD"],  # $100k starting capital
    bar_execution=False,  # Use tick data for realistic execution
)

print(f"🏢 Venue: {venue.name} (${venue.starting_balances[0]} starting balance)")

# 2. Data Configuration
# Bar data for strategy signals
bar_type_str = f"{instrument_id}-1-MINUTE-LAST-EXTERNAL"
bar_data = BacktestDataConfig(
    catalog_path=CATALOG_PATH,
    data_cls=Bar,
    instrument_id=instrument_id,
    start_time=START_TIME,
    end_time=END_TIME,
    bar_spec=[bar_type_str]
)

# Quote data for realistic execution
quote_data = BacktestDataConfig(
    catalog_path=CATALOG_PATH,
    data_cls=QuoteTick,
    instrument_id=instrument_id,
    start_time=START_TIME,
    end_time=END_TIME
)

print(f"📊 Bar data: {bar_type_str}")
print(f"💬 Quote data: QuoteTick for {instrument_id}")

# 3. Strategy Configuration
strategy_config = {
    "instrument_id": str(instrument_id),
    "bar_type": bar_type_str,
    "fast_period": 12,
    "slow_period": 26,
    "trade_size": 1  # 1 contract per trade
}

# 4. Engine Configuration
engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="__main__:MACDStrategy",
            config_path="__main__:MACDConfig",
            config=strategy_config
        )
    ],
    logging=LoggingConfig(
        log_level="WARNING",
        print_config=True
    )
)

print(f"🤖 Engine: 1 strategy configured")

# 5. Complete Run Configuration
config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[bar_data, quote_data]
)

print(f"\n✅ Backtest configuration complete!")
print(f"📋 Summary:")
print(f"  - Instrument: {instrument_id}")
print(f"  - Venue: {venue.name}")
print(f"  - Data: {len(config.data)} feeds")
print(f"  - Strategies: {len(config.engine.strategies)}")
print(f"  - Time range: {START_TIME} to {END_TIME}")


🎯 Configuring backtest for instrument: ES.n.0.GLBX
🏢 Venue: GLBX ($100_000 USD starting balance)
📊 Bar data: ES.n.0.GLBX-1-MINUTE-LAST-EXTERNAL
💬 Quote data: QuoteTick for ES.n.0.GLBX
🤖 Engine: 1 strategy configured

✅ Backtest configuration complete!
📋 Summary:
  - Instrument: ES.n.0.GLBX
  - Venue: GLBX
  - Data: 2 feeds
  - Strategies: 1
  - Time range: 2023-12-01T08:30:00Z to 2023-12-08T15:01:00Z


In [8]:
from backtest.run_backtest import run_backtest

# Create and run backtest
print("🚀 Starting backtest...")
print("⏳ This may take a few minutes depending on data size...")

try:
    # Create BacktestNode
    node = BacktestNode(configs=[config])
    
    # Run backtest with our enhanced function
    results = run_backtest(node=node)
    
    print("\n🎉 Backtest completed successfully!")
    
    # Display results summary
    for run_id, result in results.items():
        print(f"\n📊 Results for {run_id}:")
        print(result.summary())
        
        # Save results to sample_data directory
        folder_name = result.save(format='csv', base_dir='sample_data/backtest_results')
        print(f"💾 Results saved to: sample_data/backtest_results/{folder_name}")
        
except Exception as e:
    print(f"❌ Backtest failed: {e}")
    import traceback
    traceback.print_exc()
    raise

# Store result for visualization
backtest_result = list(results.values())[0]
print(f"\n📈 Ready for visualization and analysis!")

🚀 Starting backtest...
⏳ This may take a few minutes depending on data size...

🎉 Backtest completed successfully!

📊 Results for BACKTESTER-001:
Backtest Results for MACDStrategy-000
Period: 2023-12-01 08:30:00.163463+00:00 to 2023-12-08 14:33:00+00:00
Total Return: -3.56%
Sharpe Ratio: -9.33
Max Drawdown: -4.19%
Win Rate: 20.71%
Trades: 280
BacktestResult saved to sample_data/backtest_results/MACDStrategy-000_2023-12-01_08-30-00 (format: csv)
💾 Results saved to: sample_data/backtest_results/MACDStrategy-000_2023-12-01_08-30-00

📈 Ready for visualization and analysis!


C:\SquareKettle\SweTakeHome\src\backtest\run_backtest.py:254: UserWarning: Discarding nonzero nanoseconds in conversion.
  start_time = pd.to_datetime(equity_curve["timestamp"].iloc[0], utc=True).to_pydatetime()


## 6 (Optional). Add custom indicators

In [9]:
# Add custom indicators to demonstrate the visualization feature
# Simple mock data completely independent of problematic equity curve values

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("📊 Creating custom indicators with simple mock data...")

# Create simple time series (daily data for one month)
start_date = datetime(2023, 12, 1)
dates = [start_date + timedelta(days=i) for i in range(30)]

# Create custom indicators dictionary with pure mock data
custom_indicators = {}

# RSI-like indicator (oscillating between 0-100)
rsi_values = 50 + 30 * np.sin(np.linspace(0, 4*np.pi, len(dates))) + np.random.normal(0, 3, len(dates))
rsi_values = np.clip(rsi_values, 0, 100)

custom_indicators['RSI'] = pd.DataFrame({
    'timestamp': dates,
    'value': rsi_values
})

custom_indicators['BIC Indicator'] = pd.DataFrame({
    "timestamp": dates,
    "value": np.random.rand(len(dates)) * 100
})



# Add the custom indicators to the backtest result
backtest_result.custom_indicators = custom_indicators

print(f"✅ Created {len(custom_indicators)} custom indicators:")
for name, df in custom_indicators.items():
    print(f"   - {name}: {len(df)} data points")
    if not df.empty:
        value_range = f"{df['value'].min():.2f} to {df['value'].max():.2f}"
        print(f"     Range: {value_range}")

print("\n📈 Custom indicators ready for visualization!")
print("💡 These indicators will appear as an additional 'Custom Indicators' chart section")
print("🎨 Each indicator will use a different color from the Blushwave palette")

📊 Creating custom indicators with simple mock data...
✅ Created 2 custom indicators:
   - RSI: 30 data points
     Range: 18.16 to 81.24
   - BIC Indicator: 30 data points
     Range: 3.68 to 96.36

📈 Custom indicators ready for visualization!
💡 These indicators will appear as an additional 'Custom Indicators' chart section
🎨 Each indicator will use a different color from the Blushwave palette


## 7. Visualize Results!

Now we can visualize our backtest results using our custom LightweightCharts visualizer. This will launch a web-based dashboard showing:
- Interactive equity curve chart
- Price chart with trade markers 
- Performance metrics dashboard
- Position analysis

The visualizer uses TradingView's lightweight-charts library to create professional-looking financial charts.

In [10]:
PORT = 5001

# Import our custom visualizer
from viz.lightweight_charts_visualizer import LightweightChartsVisualizer, stop_flask_servers

# Execute the safe port cleanup
stop_flask_servers(PORT)

# Get the first backtest result
if isinstance(result, dict):
    # prefer an explicit key if you use one; otherwise take the first value
    backtest_result = result.get("BACKTESTER-001") or next(iter(result.values()))
else:
    backtest_result = result  # already a BacktestResult

print(f"📊 Starting Flask API Server for: {backtest_result.strategy_name}")
print(f"📈 Total Return: {backtest_result.total_return:.2%}")
print(f"📉 Max Drawdown: {backtest_result.max_drawdown:.2%}")
print(f"🎯 Win Rate: {backtest_result.win_rate:.2%}")
print(f"📋 Total Trades: {len(backtest_result.trades)}")

# Launch the Flask API server
visualizer = LightweightChartsVisualizer(port=PORT, auto_open=False)
url = visualizer.show(backtest_result)

print(f"\n✅ Flask API server is running at: {url}")
print(f"🌐 React frontend should be at: http://localhost:5001")


print("\n📡 Available API endpoints:")
print("- Equity curve: http://localhost:5001/api/equity-curve")
print("- Trades: http://localhost:5001/api/trade-markers")
print("- Trades: http://localhost:5001/api/trades-by-time-slot")
print("- Price data: http://localhost:5001/api/price-data")
print("- Metrics: http://localhost:5001/api/metrics")
print("- Positions: http://localhost:5001/api/positions")

⚠️ lsof command not available, skipping port cleanup
📊 Starting Flask API Server for: MACDStrategy-000
📈 Total Return: -3.56%
📉 Max Drawdown: -4.19%
🎯 Win Rate: 20.71%
📋 Total Trades: 280
Flask API server running at: http://127.0.0.1:5001

✅ Flask API server is running at: http://127.0.0.1:5001
🌐 React frontend should be at: http://localhost:5001

📡 Available API endpoints:
- Equity curve: http://localhost:5001/api/equity-curve
- Trades: http://localhost:5001/api/trade-markers
- Trades: http://localhost:5001/api/trades-by-time-slot
- Price data: http://localhost:5001/api/price-data
- Metrics: http://localhost:5001/api/metrics
- Positions: http://localhost:5001/api/positions


## 8. Stop the visualizer
Stop the visualizer to free up the port

In [11]:
# Execute the safe port cleanup
stop_flask_servers(PORT)

 * Serving Flask app 'viz.lightweight_charts_visualizer'
 * Debug mode: off
⚠️ lsof command not available, skipping port cleanup


## Conclusion

This comprehensive tutorial has demonstrated the complete workflow for backtesting trading strategies with real market data:

### What We Accomplished:

1. **✅ Data Pipeline**: Fetched real ES futures data from Databento
2. **✅ Data Processing**: Converted raw data to Nautilus-compatible format
3. **✅ Strategy Development**: Implemented a MACD-based trading strategy
4. **✅ Backtesting**: Ran realistic backtests with proper execution modeling
5. **✅ Analysis**: Generated comprehensive performance metrics and visualizations
6. **✅ Reporting**: Created detailed analysis reports for future reference

### Key Features Demonstrated:

- **Real Market Data**: Used actual ES futures data from CME Group
- **Professional Execution**: Simulated realistic order fills with bid/ask spreads
- **Comprehensive Analysis**: Generated equity curves, trade analysis, and position timelines
- **Extensible Framework**: Easy to modify for different strategies and instruments
- **Production Ready**: Code structure suitable for live trading implementation

### Files Created:

- `catalog/`: Nautilus data catalog with processed market data
- `backtest_results/`: Detailed backtest results in Parquet format
- `analysis_report_*/`: Comprehensive visualization reports
- `raw/`: Cached raw data files from Databento

### Next Steps:

1. **Experiment**: Try different strategy parameters and indicators
2. **Optimize**: Run parameter optimization studies
3. **Validate**: Test on different time periods and market conditions
4. **Deploy**: Consider live trading implementation with proper risk management

This framework provides a solid foundation for systematic trading strategy development and testing. Happy trading! 🚀
